# E-DAIC Multimodal Pipeline

This notebook extends the speech-only pipeline with two additional modalities:
- **Text** — linguistic features from interview transcripts
- **Facial** — action unit volatility from OpenFace

## Structure
1. Setup & imports
2. Feature extraction — Text (transcripts)
3. Feature extraction — Facial AUs
4. Load speech features (existing enriched CSVs)
5. EXPERIMENT D — Unimodal comparison (speech vs text vs facial)
6. EXPERIMENT E — Late fusion (speech + text + facial)
7. Final test evaluation — fusion model
8. Complete results summary

## 1. Setup

In [ ]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
N_SPLITS     = 5
N_SEGMENTS   = 5   # time windows for facial volatility

# !! UPDATE THIS PATH !!
BASE_SAVE   = Path.home() / 'Library' / 'CloudStorage' / \
              'your.email' / 'My Drive' / 'EDAIC'
DATA_DIR    = BASE_SAVE / 'data'
RESULTS_DIR = BASE_SAVE / 'multimodal_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NON_FEAT_COLS = ['Participant_ID', 'PHQ_Binary', 'PHQ_Score']

print('BASE_SAVE  :', BASE_SAVE)
print('RESULTS_DIR:', RESULTS_DIR)

## 2. Text Feature Extraction

Reads each participant's transcript CSV and extracts:
- **Linguistic**: TF-IDF reduced to 20 dimensions via SVD (LSA)
- **Structural**: word count, sentence count, vocabulary richness, avg sentence length
- **Temporal**: participant speaking ratio, pause count from timing

In [ ]:
def load_transcript(pid, data_dir):
    """
    Load transcript CSV for one participant.
    Returns only the PARTICIPANT turns (not the interviewer Ellie).
    """
    path = data_dir / f'{pid}_P' / f'{pid}_Transcript.csv'
    if not path.exists():
        return None

    try:
        df = pd.read_csv(path)
    except Exception:
        try:
            df = pd.read_csv(path, sep='\t')
        except Exception:
            return None

    # Keep only participant turns — column is usually 'speaker' or 'Person'
    speaker_col = next(
        (c for c in df.columns if c.lower() in ['speaker', 'person', 'who']),
        None
    )
    if speaker_col:
        # Participant is usually labelled 'Participant' or 'P'
        df = df[df[speaker_col].str.upper().str.startswith('P')]

    return df


def extract_text_features(pid, data_dir):
    """Extract structural text features for one participant."""
    df = load_transcript(pid, data_dir)
    if df is None or len(df) == 0:
        return None, None

    # Find text column
    text_col = next(
        (c for c in df.columns
         if c.lower() in ['value', 'text', 'utterance', 'transcript', 'word']),
        None
    )
    if text_col is None:
        # Fall back to last string column
        str_cols = [c for c in df.columns if df[c].dtype == object]
        text_col = str_cols[-1] if str_cols else None

    if text_col is None:
        return None, None

    utterances = df[text_col].dropna().astype(str).tolist()
    full_text  = ' '.join(utterances)
    words      = full_text.lower().split()

    # ── Structural features ────────────────────────────────────────────────────
    n_utterances   = len(utterances)
    n_words        = len(words)
    n_unique_words = len(set(words))
    vocab_richness = n_unique_words / max(n_words, 1)   # type-token ratio
    avg_utt_len    = n_words / max(n_utterances, 1)

    # ── Timing features (if start/stop columns exist) ──────────────────────────
    start_col = next((c for c in df.columns if 'start' in c.lower()), None)
    stop_col  = next(
        (c for c in df.columns if 'stop' in c.lower() or 'end' in c.lower()),
        None
    )

    speaking_ratio = 0.5  # default
    pause_rate     = 0.0

    if start_col and stop_col:
        try:
            starts   = pd.to_numeric(df[start_col], errors='coerce').dropna()
            stops    = pd.to_numeric(df[stop_col],  errors='coerce').dropna()
            durations = stops.values - starts.values[:len(stops)]
            total_dur = stops.max() - starts.min()
            speaking_ratio = durations.sum() / max(total_dur, 1)

            # Pauses between utterances
            gaps = starts.values[1:] - stops.values[:-1]
            pause_rate = (gaps > 1.0).sum() / max(n_utterances, 1)
        except Exception:
            pass

    struct_features = {
        'txt_n_utterances'  : n_utterances,
        'txt_n_words'       : n_words,
        'txt_vocab_richness': vocab_richness,
        'txt_avg_utt_len'   : avg_utt_len,
        'txt_speaking_ratio': speaking_ratio,
        'txt_pause_rate'    : pause_rate,
    }

    return full_text, struct_features


print('Text feature functions defined.')

In [ ]:
# ── Extract text features for all participants ─────────────────────────────────

def build_text_features(label_df, data_dir, split_name):
    rows      = []
    all_texts = []
    pids      = []

    for _, row in label_df.iterrows():
        pid = int(row['Participant_ID'])
        full_text, struct = extract_text_features(pid, data_dir)

        if struct is None:
            print(f'  [WARN] No transcript for {pid}')
            full_text = ''
            struct = {
                'txt_n_utterances': 0, 'txt_n_words': 0,
                'txt_vocab_richness': 0, 'txt_avg_utt_len': 0,
                'txt_speaking_ratio': 0.5, 'txt_pause_rate': 0,
            }

        row_data = {
            'Participant_ID': pid,
            'PHQ_Binary'    : int(row['PHQ_Binary']),
            'PHQ_Score'     : float(row.get('PHQ_Score', row.get('PHQ8_Score', 0))),
        }
        row_data.update(struct)
        rows.append(row_data)
        all_texts.append(full_text)
        pids.append(pid)

    return pd.DataFrame(rows), all_texts, pids


# Load label files
train_labels = pd.read_csv(BASE_SAVE / 'labels' / 'train_split.csv')
dev_labels   = pd.read_csv(BASE_SAVE / 'labels' / 'dev_split.csv')
test_labels  = pd.read_csv(BASE_SAVE / 'labels' / 'test_split.csv')

print('Extracting text features...')
train_text_df, train_texts, train_pids = build_text_features(train_labels, DATA_DIR, 'train')
dev_text_df,   dev_texts,   dev_pids   = build_text_features(dev_labels,   DATA_DIR, 'dev')
test_text_df,  test_texts,  test_pids  = build_text_features(test_labels,  DATA_DIR, 'test')

print(f'\nTrain: {len(train_text_df)} | Dev: {len(dev_text_df)} | Test: {len(test_text_df)}')
print(f'Structural text features: {len([c for c in train_text_df.columns if c.startswith("txt_")])}')

In [ ]:
# ── TF-IDF + LSA (latent semantic analysis) ───────────────────────────────────
# Reduces transcript vocabulary to 20 semantic dimensions
# Fit ONLY on train texts, transform dev and test

N_LSA_COMPONENTS = 20

tfidf = TfidfVectorizer(
    max_features  = 500,
    min_df        = 2,
    max_df        = 0.95,
    stop_words    = 'english',
    ngram_range   = (1, 2),
)

# Fit on train only
X_tfidf_train = tfidf.fit_transform(train_texts)
X_tfidf_dev   = tfidf.transform(dev_texts)
X_tfidf_test  = tfidf.transform(test_texts)

# Reduce with SVD (LSA)
n_comp = min(N_LSA_COMPONENTS, X_tfidf_train.shape[1] - 1)
svd    = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_dev   = svd.transform(X_tfidf_dev)
X_lsa_test  = svd.transform(X_tfidf_test)

# Add LSA columns to text DataFrames
lsa_cols = [f'txt_lsa_{i}' for i in range(n_comp)]

for df, X_lsa in [
    (train_text_df, X_lsa_train),
    (dev_text_df,   X_lsa_dev),
    (test_text_df,  X_lsa_test)
]:
    for j, col in enumerate(lsa_cols):
        df[col] = X_lsa[:, j]

# Save
train_text_df.to_csv(BASE_SAVE / 'text_train.csv', index=False)
dev_text_df.to_csv(BASE_SAVE   / 'text_dev.csv',   index=False)
test_text_df.to_csv(BASE_SAVE  / 'text_test.csv',  index=False)

text_feat_cols = [c for c in train_text_df.columns if c not in NON_FEAT_COLS]
print(f'Total text features: {len(text_feat_cols)}')
print(f'  Structural : 6')
print(f'  LSA (TF-IDF): {n_comp}')
print('Saved text_train/dev/test.csv')

## 3. Facial AU Feature Extraction

Reads OpenFace AU files and extracts segment-level volatility features
using the same approach as your speech pipeline — 5 time windows per interview.

In [ ]:
def extract_facial_features(pid, data_dir, n_segments=5):
    """
    Extract segment-level volatility features from OpenFace AU file.
    Features: AU intensities (r suffix) + head pose (x,y,z rotation)
    """
    path = data_dir / f'{pid}_P' / 'features' / \
           f'{pid}_OpenFace2.1.0_Pose_gaze_AUs.csv'

    if not path.exists():
        return None

    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception:
        return None

    # Keep only high-confidence frames
    if 'confidence' in df.columns:
        df = df[pd.to_numeric(df['confidence'], errors='coerce') > 0.8]

    if len(df) < n_segments:
        return None

    # Select AU intensity columns (end in _r) + head pose columns
    au_cols   = [c for c in df.columns if c.strip().endswith('_r')
                 and 'AU' in c]
    pose_cols = [c for c in df.columns
                 if c.strip() in ['pose_Rx', 'pose_Ry', 'pose_Rz']]
    feat_cols = au_cols + pose_cols

    if not feat_cols:
        return None

    features  = {}
    segments  = np.array_split(np.arange(len(df)), n_segments)

    for col in feat_cols:
        vals_all  = pd.to_numeric(df[col], errors='coerce').values
        seg_means = []
        seg_stds  = []

        for seg_idx in segments:
            seg_v = vals_all[seg_idx]
            seg_v = seg_v[~np.isnan(seg_v)]
            seg_means.append(np.mean(seg_v) if len(seg_v) > 0 else np.nan)
            seg_stds.append(np.std(seg_v)   if len(seg_v) > 0 else np.nan)

        seg_means = np.array(seg_means)
        seg_stds  = np.array(seg_stds)
        vm = seg_means[~np.isnan(seg_means)]
        vs = seg_stds[~np.isnan(seg_stds)]

        safe_col = col.strip().replace(' ', '_')
        features[f'fac_{safe_col}_seg_mean_std']   = float(np.std(vm))  if len(vm) > 1 else 0.0
        features[f'fac_{safe_col}_seg_mean_range'] = float(np.ptp(vm))  if len(vm) > 1 else 0.0
        features[f'fac_{safe_col}_seg_std_mean']   = float(np.mean(vs)) if len(vs) > 0 else 0.0
        features[f'fac_{safe_col}_seg_trend']      = float(
            np.polyfit(np.arange(len(vm)), vm, 1)[0]
        ) if len(vm) > 1 else 0.0
        all_v = vals_all[~np.isnan(vals_all)]
        features[f'fac_{safe_col}_global_std']     = float(np.std(all_v)) if len(all_v) > 0 else 0.0

    return features


def build_facial_features(label_df, data_dir, n_segments, split_name):
    rows = []
    for _, row in label_df.iterrows():
        pid  = int(row['Participant_ID'])
        feat = extract_facial_features(pid, data_dir, n_segments)

        if feat is None:
            print(f'  [WARN] No facial data for {pid}')
            feat = {}

        row_data = {
            'Participant_ID': pid,
            'PHQ_Binary'    : int(row['PHQ_Binary']),
            'PHQ_Score'     : float(row.get('PHQ_Score', row.get('PHQ8_Score', 0))),
        }
        row_data.update(feat)
        rows.append(row_data)

    df = pd.DataFrame(rows).fillna(0)
    print(f'  {split_name}: {len(df)} participants, '
          f'{len([c for c in df.columns if c.startswith("fac_")])} features')
    return df


print('Extracting facial features...')
train_fac_df = build_facial_features(train_labels, DATA_DIR, N_SEGMENTS, 'train')
dev_fac_df   = build_facial_features(dev_labels,   DATA_DIR, N_SEGMENTS, 'dev')
test_fac_df  = build_facial_features(test_labels,  DATA_DIR, N_SEGMENTS, 'test')

train_fac_df.to_csv(BASE_SAVE / 'facial_train.csv', index=False)
dev_fac_df.to_csv(BASE_SAVE   / 'facial_dev.csv',   index=False)
test_fac_df.to_csv(BASE_SAVE  / 'facial_test.csv',  index=False)

fac_feat_cols = [c for c in train_fac_df.columns if c not in NON_FEAT_COLS]
print(f'\nTotal facial features: {len(fac_feat_cols)}')
print('Saved facial_train/dev/test.csv')

## 4. Load Speech Features (existing)

In [ ]:
# =========================================================
# LOAD SPEECH FEATURES
# SPEECH_FEATURES controls which speech representation to use:
#   "enriched"  — original eGeMAPS compact features
#   "merged"    — merged E-DAIC + DAIC-WOZ segment features
#   "wav2vec2"  — deep wav2vec2 features (recommended)
# =========================================================

SPEECH_FEATURES = "wav2vec2"   # ← change this to switch

SPEECH_FILES = {
    "enriched" : ("enriched_egemap_train.csv", "enriched_egemap_dev.csv",  "enriched_egemap_test.csv"),
    "merged"   : ("merged_segment_train.csv",   "merged_segment_dev.csv",   "merged_segment_test.csv"),
    "wav2vec2" : ("wav2vec2_segment_train.csv", "wav2vec2_segment_dev.csv", "wav2vec2_segment_test.csv"),
}

tr_f, dv_f, te_f = SPEECH_FILES[SPEECH_FEATURES]
speech_train = pd.read_csv(BASE_SAVE / tr_f)
speech_dev   = pd.read_csv(BASE_SAVE / dv_f)
speech_test  = pd.read_csv(BASE_SAVE / te_f)

speech_feat_cols = [c for c in speech_train.columns if c not in NON_FEAT_COLS]

print(f"Speech features : {SPEECH_FEATURES}")
print(f"  Train: {len(speech_train)} | Dev: {len(speech_dev)} | Test: {len(speech_test)}")
print(f"  Feature count: {len(speech_feat_cols)}")
if "source" in speech_train.columns:
    print(f"  Sources: {speech_train['source'].value_counts().to_dict()}")


## 5. Helper functions for multimodal CV

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'UAR'     : recall_score(y_true, y_pred, average='macro'),
        'F1'      : f1_score(y_true, y_pred, average='weighted'),
    }


def prepare_fold(X_tr, y_tr, X_va, *, random_state=42):
    """Impute + scale inside fold only. No leakage."""
    imp = SimpleImputer(strategy='mean')
    X_tr = imp.fit_transform(X_tr)
    X_va = imp.transform(X_va)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)
    return X_tr, y_tr, X_va


def run_unimodal_cv(X_full, y_full, model, model_name, modality_name,
                    n_splits=5, random_state=42):
    """Run stratified CV for one modality + model combination."""
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True,
                           random_state=random_state)
    rows = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full), 1):
        X_tr, y_tr, X_va = prepare_fold(
            X_full[tr_idx], y_full[tr_idx], X_full[va_idx],
            random_state=random_state + fold
        )
        clf = clone(model)
        clf.fit(X_tr, y_tr)
        m = compute_metrics(y_full[va_idx], clf.predict(X_va))
        rows.append({
            'modality': modality_name, 'model': model_name,
            'fold': fold, **m
        })
    return pd.DataFrame(rows)


def get_fold_probabilities(X_full, y_full, model, n_splits=5, random_state=42):
    """
    Get out-of-fold probability predictions for late fusion.
    Returns array of shape (n_samples,) with P(depressed) for each sample.
    """
    skf   = StratifiedKFold(n_splits=n_splits, shuffle=True,
                            random_state=random_state)
    probs = np.zeros(len(y_full))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full), 1):
        X_tr, y_tr, X_va = prepare_fold(
            X_full[tr_idx], y_full[tr_idx], X_full[va_idx],
            random_state=random_state + fold
        )
        clf = clone(model)
        clf.fit(X_tr, y_tr)
        probs[va_idx] = clf.predict_proba(X_va)[:, 1]

    return probs


print('Multimodal helper functions defined.')

## 6. Prepare combined train+dev arrays

In [ ]:
# Combine train + dev for CV
speech_traindev = pd.concat([speech_train, speech_dev], ignore_index=True)
text_traindev   = pd.concat([train_text_df, dev_text_df], ignore_index=True)
facial_traindev = pd.concat([train_fac_df,  dev_fac_df],  ignore_index=True)

# Labels (use speech as reference — all splits have same participants)
y_full = speech_traindev['PHQ_Binary'].values
y_test = speech_test['PHQ_Binary'].values

# Feature arrays
X_speech_full = speech_traindev[speech_feat_cols].values
X_text_full   = text_traindev[text_feat_cols].values
X_facial_full = facial_traindev[fac_feat_cols].values

X_speech_test = speech_test[speech_feat_cols].values
X_text_test   = test_text_df[text_feat_cols].values
X_facial_test = test_fac_df[fac_feat_cols].values

print(f'Train+Dev samples : {len(y_full)}')
print(f'Test samples      : {len(y_test)}')
print(f'Speech features   : {X_speech_full.shape[1]}')
print(f'Text features     : {X_text_full.shape[1]}')
print(f'Facial features   : {X_facial_full.shape[1]}')

## 7. EXPERIMENT D — Unimodal comparison

Each modality tested independently with the same SVM classifier.
This shows the contribution of each modality before fusion.

In [ ]:
SVM_MODEL = SVC(
    C=10, kernel='rbf', gamma='scale',
    class_weight='balanced',
    probability=True,           # needed for late fusion
    random_state=RANDOM_STATE
)

unimodal_results = []

for modality_name, X_full in [
    ('Speech', X_speech_full),
    ('Text',   X_text_full),
    ('Facial', X_facial_full),
]:
    print(f'Running {modality_name} CV...')
    df = run_unimodal_cv(
        X_full, y_full, SVM_MODEL, 'SVM', modality_name,
        n_splits=N_SPLITS, random_state=RANDOM_STATE
    )
    unimodal_results.append(df)
    mean_uar = df['UAR'].mean()
    print(f'  UAR: {mean_uar:.4f} ± {df["UAR"].std():.4f}')

unimodal_df = pd.concat(unimodal_results, ignore_index=True)
unimodal_df.to_csv(RESULTS_DIR / 'unimodal_cv_results.csv', index=False)

unimodal_summary = (
    unimodal_df.groupby('modality')[['accuracy', 'UAR', 'F1']]
    .agg(['mean', 'std']).round(4)
)
print('\n===== EXPERIMENT D: Unimodal Comparison =====')
display(unimodal_summary)

## 8. EXPERIMENT E — Late Fusion

Each modality produces a probability score P(depressed).
Fusion = weighted average of the three probability scores.
Weights are proportional to each modality's CV UAR.

In [ ]:
# ── Get out-of-fold probabilities for each modality ───────────────────────────
print('Getting out-of-fold probabilities for fusion...')

probs_speech = get_fold_probabilities(
    X_speech_full, y_full, SVM_MODEL, N_SPLITS, RANDOM_STATE
)
print('  Speech done')

probs_text = get_fold_probabilities(
    X_text_full, y_full, SVM_MODEL, N_SPLITS, RANDOM_STATE
)
print('  Text done')

probs_facial = get_fold_probabilities(
    X_facial_full, y_full, SVM_MODEL, N_SPLITS, RANDOM_STATE
)
print('  Facial done')

# ── Compute fusion weights from CV UAR ────────────────────────────────────────
uar_speech = unimodal_df[unimodal_df['modality']=='Speech']['UAR'].mean()
uar_text   = unimodal_df[unimodal_df['modality']=='Text']['UAR'].mean()
uar_facial = unimodal_df[unimodal_df['modality']=='Facial']['UAR'].mean()

total_uar  = uar_speech + uar_text + uar_facial
w_speech   = uar_speech / total_uar
w_text     = uar_text   / total_uar
w_facial   = uar_facial / total_uar

print(f'\nFusion weights:')
print(f'  Speech : {w_speech:.3f} (UAR={uar_speech:.4f})')
print(f'  Text   : {w_text:.3f}   (UAR={uar_text:.4f})')
print(f'  Facial : {w_facial:.3f} (UAR={uar_facial:.4f})')

In [ ]:
# ── Evaluate fusion on out-of-fold predictions ────────────────────────────────
# Try three fusion strategies

fusion_results = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_speech_full, y_full), 1):
    y_va = y_full[va_idx]

    p_s = probs_speech[va_idx]
    p_t = probs_text[va_idx]
    p_f = probs_facial[va_idx]

    # Strategy 1: Simple average
    fused_avg   = (p_s + p_t + p_f) / 3
    preds_avg   = (fused_avg >= 0.5).astype(int)
    m_avg = compute_metrics(y_va, preds_avg)

    # Strategy 2: Weighted average (by UAR)
    fused_w     = w_speech * p_s + w_text * p_t + w_facial * p_f
    preds_w     = (fused_w >= 0.5).astype(int)
    m_w = compute_metrics(y_va, preds_w)

    # Strategy 3: Speech + Text only (best two modalities)
    fused_st    = 0.5 * p_s + 0.5 * p_t
    preds_st    = (fused_st >= 0.5).astype(int)
    m_st = compute_metrics(y_va, preds_st)

    for strategy, m in [
        ('Fusion (equal weights)',    m_avg),
        ('Fusion (UAR weights)',      m_w),
        ('Fusion (Speech+Text only)', m_st),
    ]:
        fusion_results.append({
            'strategy': strategy, 'fold': fold, **m
        })

fusion_df = pd.DataFrame(fusion_results)
fusion_df.to_csv(RESULTS_DIR / 'fusion_cv_results.csv', index=False)

fusion_summary = (
    fusion_df.groupby('strategy')[['accuracy', 'UAR', 'F1']]
    .agg(['mean', 'std']).round(4)
)
print('\n===== EXPERIMENT E: Late Fusion CV Results =====')
display(fusion_summary)

## 9. Final held-out test evaluation

In [ ]:
# Train on FULL train+dev, evaluate on untouched test set

def train_and_predict(X_full, y_full, X_test, model):
    imp    = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(imp.fit_transform(X_full))
    X_te   = scaler.transform(imp.transform(X_test))
    clf    = clone(model)
    clf.fit(X_tr, y_full)
    probs  = clf.predict_proba(X_te)[:, 1]
    preds  = clf.predict(X_te)
    return probs, preds


print('Training final models on full train+dev...')

probs_speech_test, preds_speech_test = train_and_predict(
    X_speech_full, y_full, X_speech_test, SVM_MODEL
)
probs_text_test, preds_text_test = train_and_predict(
    X_text_full, y_full, X_text_test, SVM_MODEL
)
probs_facial_test, preds_facial_test = train_and_predict(
    X_facial_full, y_full, X_facial_test, SVM_MODEL
)

# Best fusion strategy from CV
best_strategy = fusion_df.groupby('strategy')['UAR'].mean().idxmax()
print(f'Best fusion strategy from CV: {best_strategy}')

if 'Speech+Text' in best_strategy:
    fused_test = 0.5 * probs_speech_test + 0.5 * probs_text_test
elif 'UAR weights' in best_strategy:
    fused_test = (w_speech * probs_speech_test +
                  w_text   * probs_text_test +
                  w_facial * probs_facial_test)
else:
    fused_test = (probs_speech_test + probs_text_test + probs_facial_test) / 3

preds_fusion_test = (fused_test >= 0.5).astype(int)

# Collect test results
test_rows = []
for name, preds in [
    ('Speech only',    preds_speech_test),
    ('Text only',      preds_text_test),
    ('Facial only',    preds_facial_test),
    (best_strategy,    preds_fusion_test),
]:
    m = compute_metrics(y_test, preds)
    test_rows.append({'Model': name, **m})

test_results = pd.DataFrame(test_rows).round(4)
test_results = test_results.sort_values('UAR', ascending=False)

print('\n===== FINAL HELD-OUT TEST RESULTS =====')
display(test_results)
test_results.to_csv(RESULTS_DIR / 'multimodal_test_results.csv', index=False)

In [ ]:
# ── Confusion matrices ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, preds) in zip(axes, [
    ('Speech only',  preds_speech_test),
    ('Text only',    preds_text_test),
    ('Facial only',  preds_facial_test),
    ('Fusion',       preds_fusion_test),
]):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Not Dep.', 'Dep.']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    uar = recall_score(y_test, preds, average='macro')
    ax.set_title(f'{name}\nUAR={uar:.3f}')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'multimodal_confusion_matrices.png', dpi=150)
plt.show()
print('Saved: multimodal_confusion_matrices.png')

In [ ]:
# ── Final summary chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# CV results
cv_summary = pd.concat([
    unimodal_df.groupby('modality')['UAR'].agg(['mean','std']).rename(
        index=lambda x: f'{x} (CV)'
    ),
    fusion_df.groupby('strategy')['UAR'].agg(['mean','std']).rename(
        index=lambda x: f'{x} (CV)'
    )
]).sort_values('mean')

colors = ['steelblue' if 'Fusion' in i else
          'mediumseagreen' if 'Text' in i else
          'coral' if 'Facial' in i else
          'lightgrey'
          for i in cv_summary.index]

bars = ax.barh(cv_summary.index, cv_summary['mean'],
               xerr=cv_summary['std'], color=colors,
               capsize=4, alpha=0.85, edgecolor='black', linewidth=0.5)

ax.axvline(0.5, linestyle='--', color='grey', linewidth=1.2, label='Chance (0.5)')
ax.set_xlabel('UAR (mean ± std over 5 folds)')
ax.set_title('Multimodal Comparison — Unweighted Average Recall (UAR)')
ax.set_xlim(0, 1)
ax.legend()

for bar, val in zip(bars, cv_summary['mean']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'multimodal_uar_chart.png', dpi=150)
plt.show()
print('Saved: multimodal_uar_chart.png')
print('\nAll multimodal results saved to:', RESULTS_DIR)